In [1]:
import pandas as pd

# Leer cada pestaña del archivo Excel en un DataFrame
df_potencial = pd.read_excel("Datasets.xlsx", sheet_name="Potencial")
df_clientes = pd.read_excel("Datasets.xlsx", sheet_name="Clientes")
df_clientes = df_clientes.drop(columns=['Unnamed: 1', 'Provincia'])
df_productos = pd.read_excel("Datasets.xlsx", sheet_name="Productos")
df_ventas = pd.read_excel("Datasets.xlsx", sheet_name="Ventas")
df_campanas = pd.read_excel("Datasets.xlsx", sheet_name="Campañas")

print(f"Potencial: {df_potencial.shape}")
print(f"Clientes:  {df_clientes.shape}")
print(f"Productos: {df_productos.shape}")
print(f"Ventas:    {df_ventas.shape}")
print(f"Campañas:  {df_campanas.shape}")

Potencial: (33093, 4)
Clientes:  (11031, 1)
Productos: (25, 4)
Ventas:    (162546, 6)
Campañas:  (10, 3)


In [2]:
# Ordenar ventas validas por cliente, producto y fecha
df_ventas_validas = df_ventas[
    (df_ventas['Unidades'] > 0)
    & (df_ventas['Valores_H'] > 0)
].copy()
df_sorted = df_ventas_validas.sort_values(['Id. Cliente', 'Id. Producto', 'Fecha'])

# Calcular días entre compras consecutivas del mismo producto por el mismo cliente
df_sorted['dias_entre_compras'] = (
    df_sorted.groupby(['Id. Cliente', 'Id. Producto'])['Fecha']
    .diff()
    .dt.days
)

# Agregar globalmente por producto: media y desviación estándar en días
stats_producto = (
    df_sorted.dropna(subset=['dias_entre_compras'])
    .groupby('Id. Producto')['dias_entre_compras']
    .agg(
        tiempo_medio_recompra_dias='mean',
        std_recompra_dias='std'
    )
    .reset_index()
)

# Unir a df_productos
df_productos = df_productos.merge(
    stats_producto,
    left_on='Id.Prod',
    right_on='Id. Producto',
    how='left'
).drop(columns=['Id. Producto'])

df_productos[['Id.Prod', 'Bloque analítico', 'Categoria_H', 'Familia_H', 'tiempo_medio_recompra_dias', 'std_recompra_dias']]

,Id.Prod,Bloque analítico,Categoria_H,Familia_H,tiempo_medio_recompra_dias,std_recompra_dias
0,4565,Commodities,Categoria C1,Familia C1,173.383726,161.061119
1,4566,Commodities,Categoria C1,Familia C1,158.404580,157.536304
2,3399,Commodities,Categoria C1,Familia C1,304.200549,264.305039
3,178,Commodities,Categoria C1,Familia C1,206.248285,185.936005
4,4567,Commodities,Categoria C1,Familia C1,299.145040,280.055722
5,3408,Commodities,Categoria C1,Familia C1,184.320516,165.241975
6,4911,Commodities,Categoria C2,Familia C2,206.847581,212.449632
7,4912,Commodities,Categoria C2,Familia C2,134.174448,152.837632
8,5100,Productos Técnicos,Categoria T1,Familia T1,95.261593,148.223712
9,5099,Productos Técnicos,Categoria T1,Familia T1,108.254921,145.801652


In [3]:
# Tabla cliente-producto con frecuencia, recompra y recencia
# Se consideran compras solo las líneas con unidades y valor positivos; los negativos suelen ser abonos/devoluciones.
ventas_compras = df_ventas[
    (df_ventas['Unidades'] > 0)
    & (df_ventas['Valores_H'] > 0)
].copy()
ventas_compras['Fecha'] = pd.to_datetime(ventas_compras['Fecha'])

# Una línea de factura cuenta como una compra. Si prefieres contar días únicos, sustituye Num.Fact por Fecha.
ventas_ordenadas = ventas_compras.sort_values(['Id. Cliente', 'Id. Producto', 'Fecha'])
ventas_ordenadas['dias_entre_compras'] = (
    ventas_ordenadas.groupby(['Id. Cliente', 'Id. Producto'])['Fecha']
    .diff()
    .dt.days
)

fecha_referencia = ventas_ordenadas['Fecha'].max()

tabla_cliente_producto = (
    ventas_ordenadas
    .groupby(['Id. Cliente', 'Id. Producto'])
    .agg(
        veces_comprado=('Num.Fact', 'count'),
        tiempo_medio_entre_compras_dias=('dias_entre_compras', 'mean'),
        std_entre_compras_dias=('dias_entre_compras', 'std'),
        ultima_compra=('Fecha', 'max'),
    )
    .reset_index()
)

tabla_cliente_producto['dias_desde_ultima_compra'] = (
    fecha_referencia - tabla_cliente_producto['ultima_compra']
).dt.days

std_cliente_producto = tabla_cliente_producto['std_entre_compras_dias'].mask(
    tabla_cliente_producto['std_entre_compras_dias'] == 0
)
tabla_cliente_producto['zscore_recencia_cliente_producto'] = (
    tabla_cliente_producto['dias_desde_ultima_compra']
    - tabla_cliente_producto['tiempo_medio_entre_compras_dias']
) / std_cliente_producto

tabla_cliente_producto = (
    tabla_cliente_producto
    .merge(df_clientes, on='Id. Cliente', how='left')
    .merge(
        df_productos,
        left_on='Id. Producto',
        right_on='Id.Prod',
        how='left'
    )
    .drop(columns=['Id.Prod'])
    .sort_values(['Id. Cliente', 'Id. Producto'])
    .reset_index(drop=True)
)

std_recompra_general = tabla_cliente_producto['std_recompra_dias'].mask(
    tabla_cliente_producto['std_recompra_dias'] == 0
)
tabla_cliente_producto['zscore_recencia_recompra_general'] = (
    tabla_cliente_producto['dias_desde_ultima_compra']
    - tabla_cliente_producto['tiempo_medio_recompra_dias']
) / std_recompra_general

tabla_cliente_producto


,Id. Cliente,Id. Producto,veces_comprado,tiempo_medio_entre_compras_dias,std_entre_compras_dias,ultima_compra,dias_desde_ultima_compra,zscore_recencia_cliente_producto,Bloque analítico,Categoria_H,Familia_H,tiempo_medio_recompra_dias,std_recompra_dias,zscore_recencia_recompra_general
0,3,4566,2,812.000000,NaN,2024-02-23,675,NaN,Commodities,Categoria C1,Familia C1,158.404580,157.536304,3.279215
1,3,5115,2,164.000000,NaN,2022-02-21,1407,NaN,Productos Técnicos,Categoria T1,Familia T2,123.555416,164.167315,7.817906
2,3,5116,2,164.000000,NaN,2022-02-21,1407,NaN,Productos Técnicos,Categoria T1,Familia T2,169.756052,209.511646,5.905371
3,26,3408,4,448.333333,226.954474,2025-12-02,27,-1.856466,Commodities,Categoria C1,Familia C1,184.320516,165.241975,-0.952061
4,26,4566,2,1040.000000,NaN,2025-01-31,332,NaN,Commodities,Categoria C1,Familia C1,158.404580,157.536304,1.101939
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30216,1000102310,4567,1,NaN,NaN,2025-12-16,13,NaN,Commodities,Categoria C1,Familia C1,299.145040,280.055722,-1.021743
30217,1000102318,3408,1,NaN,NaN,2025-12-22,7,NaN,Commodities,Categoria C1,Familia C1,184.320516,165.241975,-1.073096
30218,1000102321,4565,1,NaN,NaN,2025-12-22,7,NaN,Commodities,Categoria C1,Familia C1,173.383726,161.061119,-1.033047
30219,1000102321,4566,1,NaN,NaN,2025-12-22,7,NaN,Commodities,Categoria C1,Familia C1,158.404580,157.536304,-0.961077


In [4]:
# Serie temporal por cliente-producto con cada compra valida y z-scores en ese momento
compras_validas = df_ventas[
    (df_ventas['Unidades'] > 0)
    & (df_ventas['Valores_H'] > 0)
].copy()
compras_validas['Fecha'] = pd.to_datetime(compras_validas['Fecha'])

serie_temporal_cliente_producto = compras_validas.sort_values(
    ['Id. Cliente', 'Id. Producto', 'Fecha', 'Num.Fact']
).copy()
serie_temporal_cliente_producto['fecha_compra_anterior'] = (
    serie_temporal_cliente_producto
    .groupby(['Id. Cliente', 'Id. Producto'])['Fecha']
    .shift()
)
serie_temporal_cliente_producto['dias_desde_compra_anterior'] = (
    serie_temporal_cliente_producto['Fecha']
    - serie_temporal_cliente_producto['fecha_compra_anterior']
).dt.days
serie_temporal_cliente_producto['numero_compra_cliente_producto'] = (
    serie_temporal_cliente_producto
    .groupby(['Id. Cliente', 'Id. Producto'])
    .cumcount()
    + 1
)

columnas_contexto = [
    'Id. Cliente',
    'Id. Producto',
    'veces_comprado',
    'tiempo_medio_entre_compras_dias',
    'std_entre_compras_dias',
    'ultima_compra',
    'dias_desde_ultima_compra',
    'zscore_recencia_cliente_producto',
    'Bloque analítico',
    'Categoria_H',
    'Familia_H',
    'tiempo_medio_recompra_dias',
    'std_recompra_dias',
    'zscore_recencia_recompra_general',
]

serie_temporal_cliente_producto = serie_temporal_cliente_producto.merge(
    tabla_cliente_producto[columnas_contexto],
    on=['Id. Cliente', 'Id. Producto'],
    how='left'
)

std_momento_cliente_producto = serie_temporal_cliente_producto['std_entre_compras_dias'].mask(
    serie_temporal_cliente_producto['std_entre_compras_dias'] == 0
)
serie_temporal_cliente_producto['zscore_momento_cliente_producto'] = (
    serie_temporal_cliente_producto['dias_desde_compra_anterior']
    - serie_temporal_cliente_producto['tiempo_medio_entre_compras_dias']
) / std_momento_cliente_producto

std_momento_recompra_general = serie_temporal_cliente_producto['std_recompra_dias'].mask(
    serie_temporal_cliente_producto['std_recompra_dias'] == 0
)
serie_temporal_cliente_producto['zscore_momento_recompra_general'] = (
    serie_temporal_cliente_producto['dias_desde_compra_anterior']
    - serie_temporal_cliente_producto['tiempo_medio_recompra_dias']
) / std_momento_recompra_general

serie_temporal_cliente_producto = serie_temporal_cliente_producto.sort_values(
    ['Id. Cliente', 'Id. Producto', 'Fecha', 'Num.Fact']
).reset_index(drop=True)

serie_temporal_cliente_producto


,Num.Fact,Fecha,Id. Cliente,Id. Producto,Unidades,Valores_H,fecha_compra_anterior,dias_desde_compra_anterior,numero_compra_cliente_producto,veces_comprado,...,dias_desde_ultima_compra,zscore_recencia_cliente_producto,Bloque analítico,Categoria_H,Familia_H,tiempo_medio_recompra_dias,std_recompra_dias,zscore_recencia_recompra_general,zscore_momento_cliente_producto,zscore_momento_recompra_general
0,9310372336,2021-12-03,3,4566,15,1240.0488,NaT,NaN,1,2,...,675,NaN,Commodities,Categoria C1,Familia C1,158.404580,157.536304,3.279215,NaN,NaN
1,9100066916,2024-02-23,3,4566,15,1290.0690,2021-12-03,812.0,2,2,...,675,NaN,Commodities,Categoria C1,Familia C1,158.404580,157.536304,3.279215,NaN,4.148856
2,9310362839,2021-09-10,3,5115,2,872.6688,NaT,NaN,1,2,...,1407,NaN,Productos Técnicos,Categoria T1,Familia T2,123.555416,164.167315,7.817906,NaN,NaN
3,9310379175,2022-02-21,3,5115,3,1309.0032,2021-09-10,164.0,2,2,...,1407,NaN,Productos Técnicos,Categoria T1,Familia T2,123.555416,164.167315,7.817906,NaN,0.246362
4,9310362839,2021-09-10,3,5116,1,614.8120,NaT,NaN,1,2,...,1407,NaN,Productos Técnicos,Categoria T1,Familia T2,169.756052,209.511646,5.905371,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157382,9100126016,2025-12-16,1000102310,4567,1,150.9712,NaT,NaN,1,1,...,13,NaN,Commodities,Categoria C1,Familia C1,299.145040,280.055722,-1.021743,NaN,NaN
157383,9100126347,2025-12-22,1000102318,3408,6,439.7884,NaT,NaN,1,1,...,7,NaN,Commodities,Categoria C1,Familia C1,184.320516,165.241975,-1.073096,NaN,NaN
157384,9100126349,2025-12-22,1000102321,4565,6,620.0872,NaT,NaN,1,1,...,7,NaN,Commodities,Categoria C1,Familia C1,173.383726,161.061119,-1.033047,NaN,NaN
157385,9100126349,2025-12-22,1000102321,4566,18,1860.2930,NaT,NaN,1,1,...,7,NaN,Commodities,Categoria C1,Familia C1,158.404580,157.536304,-0.961077,NaN,NaN


In [5]:
# Ventas enriquecidas sin modificar df_ventas
# El potencial se enlaza por Categoria_H porque es la clave comun entre df_potencial y df_productos.
ventas_enriquecidas = df_ventas.copy()
ventas_enriquecidas['_orden_original'] = range(len(ventas_enriquecidas))
ventas_enriquecidas['Fecha'] = pd.to_datetime(ventas_enriquecidas['Fecha'])

columnas_producto = [
    'Id.Prod',
    'Bloque analítico',
    'Categoria_H',
    'Familia_H',
    'tiempo_medio_recompra_dias',
    'std_recompra_dias',
]
ventas_enriquecidas = ventas_enriquecidas.merge(
    df_productos[columnas_producto],
    left_on='Id. Producto',
    right_on='Id.Prod',
    how='left'
).drop(columns=['Id.Prod'])

ventas_enriquecidas = ventas_enriquecidas.sort_values(
    ['Id. Cliente', 'Id. Producto', 'Fecha', 'Num.Fact', '_orden_original']
).copy()
ventas_enriquecidas['_es_compra_valida'] = (
    (ventas_enriquecidas['Unidades'] > 0)
    & (ventas_enriquecidas['Valores_H'] > 0)
)
ventas_enriquecidas['_fecha_compra_valida'] = ventas_enriquecidas['Fecha'].where(
    ventas_enriquecidas['_es_compra_valida']
)
ventas_enriquecidas['_compra_valida_int'] = ventas_enriquecidas['_es_compra_valida'].astype(int)
ventas_enriquecidas['numero_compras_anteriores_producto'] = (
    ventas_enriquecidas
    .groupby(['Id. Cliente', 'Id. Producto'])['_compra_valida_int']
    .cumsum()
    - ventas_enriquecidas['_compra_valida_int']
)
compras_cliente_producto = (
    ventas_enriquecidas
    .groupby(['Id. Cliente', 'Id. Producto'])['_compra_valida_int']
    .transform('sum')
)
compras_cliente = (
    ventas_enriquecidas
    .groupby('Id. Cliente')['_compra_valida_int']
    .transform('sum')
)
ventas_enriquecidas['total_compras_cliente_otros_productos'] = (
    compras_cliente - compras_cliente_producto
)
ventas_enriquecidas['_es_devolucion_int'] = (
    ventas_enriquecidas['Valores_H'] <= 0
).astype(int)
ventas_enriquecidas['numero_devoluciones_producto'] = (
    ventas_enriquecidas
    .groupby(['Id. Cliente', 'Id. Producto'])['_es_devolucion_int']
    .transform('sum')
)
ventas_enriquecidas['fecha_compra_anterior_producto'] = (
    ventas_enriquecidas
    .groupby(['Id. Cliente', 'Id. Producto'])['_fecha_compra_valida']
    .transform(lambda s: s.shift().ffill())
)
ventas_enriquecidas['dias_desde_compra_anterior_producto'] = (
    ventas_enriquecidas['Fecha']
    - ventas_enriquecidas['fecha_compra_anterior_producto']
).dt.days

metricas_cliente_producto = tabla_cliente_producto[[
    'Id. Cliente',
    'Id. Producto',
    'tiempo_medio_entre_compras_dias',
    'std_entre_compras_dias',
]].drop_duplicates(['Id. Cliente', 'Id. Producto'])
ventas_enriquecidas = ventas_enriquecidas.merge(
    metricas_cliente_producto,
    on=['Id. Cliente', 'Id. Producto'],
    how='left'
)

std_cliente_producto = ventas_enriquecidas['std_entre_compras_dias'].mask(
    ventas_enriquecidas['std_entre_compras_dias'] == 0
)
ventas_enriquecidas['zscore_momento_cliente_producto'] = (
    ventas_enriquecidas['dias_desde_compra_anterior_producto']
    - ventas_enriquecidas['tiempo_medio_entre_compras_dias']
) / std_cliente_producto

std_recompra_general = ventas_enriquecidas['std_recompra_dias'].mask(
    ventas_enriquecidas['std_recompra_dias'] == 0
)
ventas_enriquecidas['zscore_momento_recompra_general'] = (
    ventas_enriquecidas['dias_desde_compra_anterior_producto']
    - ventas_enriquecidas['tiempo_medio_recompra_dias']
) / std_recompra_general

potencial_cliente_categoria = (
    df_potencial
    .rename(columns={
        'Id.Cliente': 'Id. Cliente',
        'Categoria Productos': 'Categoria_H',
        'Familia': 'Familia_potencial',
    })
    .groupby(['Id. Cliente', 'Categoria_H'], as_index=False)
    .agg(
        potencial_cliente_categoria=('Potencial_H', 'sum'),
        familias_potencial=('Familia_potencial', lambda s: ', '.join(sorted(s.dropna().astype(str).unique())))
    )
)

ventas_validas_potencial = ventas_enriquecidas[ventas_enriquecidas['_es_compra_valida']].copy()
ventas_validas_potencial['anio'] = ventas_validas_potencial['Fecha'].dt.year

anios_cliente_categoria = (
    ventas_validas_potencial
    .groupby(['Id. Cliente', 'Categoria_H'], as_index=False)
    .agg(n_anios_cliente_categoria=('anio', 'nunique'))
)

gasto_producto_categoria = (
    ventas_validas_potencial
    .groupby(['Id. Cliente', 'Categoria_H', 'Id. Producto'], as_index=False)
    .agg(gasto_total_cliente_categoria_producto=('Valores_H', 'sum'))
    .merge(anios_cliente_categoria, on=['Id. Cliente', 'Categoria_H'], how='left')
)
gasto_producto_categoria['gasto_medio_anual_cliente_categoria_producto'] = (
    gasto_producto_categoria['gasto_total_cliente_categoria_producto']
    / gasto_producto_categoria['n_anios_cliente_categoria'].mask(gasto_producto_categoria['n_anios_cliente_categoria'] == 0)
)

gasto_cliente_categoria = (
    ventas_validas_potencial
    .groupby(['Id. Cliente', 'Categoria_H'], as_index=False)
    .agg(gasto_total_cliente_categoria=('Valores_H', 'sum'))
    .merge(anios_cliente_categoria, on=['Id. Cliente', 'Categoria_H'], how='left')
)
gasto_cliente_categoria['gasto_medio_anual_cliente_categoria'] = (
    gasto_cliente_categoria['gasto_total_cliente_categoria']
    / gasto_cliente_categoria['n_anios_cliente_categoria'].mask(gasto_cliente_categoria['n_anios_cliente_categoria'] == 0)
)

gasto_producto_categoria = gasto_producto_categoria.merge(
    gasto_cliente_categoria[[
        'Id. Cliente',
        'Categoria_H',
        'gasto_total_cliente_categoria',
        'gasto_medio_anual_cliente_categoria',
    ]],
    on=['Id. Cliente', 'Categoria_H'],
    how='left'
)
gasto_producto_categoria['peso_producto_en_categoria'] = (
    gasto_producto_categoria['gasto_medio_anual_cliente_categoria_producto']
    / gasto_producto_categoria['gasto_medio_anual_cliente_categoria'].mask(gasto_producto_categoria['gasto_medio_anual_cliente_categoria'] == 0)
)
gasto_producto_categoria = gasto_producto_categoria.merge(
    potencial_cliente_categoria,
    on=['Id. Cliente', 'Categoria_H'],
    how='left'
)
gasto_producto_categoria['potencial_asignado_producto'] = (
    gasto_producto_categoria['potencial_cliente_categoria']
    * gasto_producto_categoria['peso_producto_en_categoria']
)

ventas_enriquecidas = ventas_enriquecidas.merge(
    gasto_producto_categoria[[
        'Id. Cliente',
        'Categoria_H',
        'Id. Producto',
        'n_anios_cliente_categoria',
        'gasto_total_cliente_categoria_producto',
        'gasto_total_cliente_categoria',
        'gasto_medio_anual_cliente_categoria_producto',
        'gasto_medio_anual_cliente_categoria',
        'peso_producto_en_categoria',
        'potencial_cliente_categoria',
        'familias_potencial',
        'potencial_asignado_producto',
    ]],
    on=['Id. Cliente', 'Categoria_H', 'Id. Producto'],
    how='left'
)

ventas_enriquecidas = (
    ventas_enriquecidas
    .drop(columns=['_fecha_compra_valida', '_es_compra_valida', '_compra_valida_int', '_es_devolucion_int'])
    .sort_values('_orden_original')
    .drop(columns=['_orden_original'])
    .reset_index(drop=True)
)

ventas_enriquecidas[['Num.Fact', 'Fecha', 'Id. Cliente', 'Id. Producto', 'Unidades',
       'Valores_H', 'Familia_H', 'tiempo_medio_recompra_dias', 'std_recompra_dias',
       'numero_compras_anteriores_producto', 'total_compras_cliente_otros_productos', 'numero_devoluciones_producto',
       'dias_desde_compra_anterior_producto', 'zscore_momento_cliente_producto', 'zscore_momento_recompra_general',
       'gasto_medio_anual_cliente_categoria_producto', 'peso_producto_en_categoria', 'potencial_asignado_producto']]

,Num.Fact,Fecha,Id. Cliente,Id. Producto,Unidades,Valores_H,Familia_H,tiempo_medio_recompra_dias,std_recompra_dias,numero_compras_anteriores_producto,total_compras_cliente_otros_productos,numero_devoluciones_producto,dias_desde_compra_anterior_producto,zscore_momento_cliente_producto,zscore_momento_recompra_general,gasto_medio_anual_cliente_categoria_producto,peso_producto_en_categoria,potencial_asignado_producto
0,3810000197,2024-01-03,1000078762,4566,-1,-141.9280,Familia C1,158.404580,157.536304,3,28,1,22.0,-5.038136,-0.865861,144.954960,0.132841,239.428126
1,3810000198,2024-01-12,1000077124,3408,-10,-634.1544,Familia C1,184.320516,165.241975,15,24,1,22.0,-2.220237,-0.982320,2964.373520,0.955389,3257.915947
2,3810000198,2024-01-12,1000077124,5099,-1,-329.3232,Familia T1,108.254921,145.801652,7,41,1,22.0,-1.699244,-0.591591,775.375900,0.244796,2263.702921
3,3810000200,2024-02-23,27995,4911,-60,-3931.9080,Familia C2,206.847581,212.449632,42,96,2,21.0,-0.539885,-0.874784,12317.441280,0.558395,1241.378487
4,3810000201,2024-03-22,35333,5099,-4,-1142.1122,Familia T1,108.254921,145.801652,1,32,1,25.0,NaN,-0.571015,228.422440,0.014121,391.752355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162541,9310439486,2023-12-28,33953,4912,24,756.6144,Familia C2,134.174448,152.837632,22,98,0,90.0,1.575228,-0.289029,5780.827920,0.692240,1923.664802
162542,9310439492,2023-12-28,41637,5112,2,688.4450,Familia T2,114.293017,154.688694,0,7,0,NaN,NaN,NaN,229.481667,0.308829,5711.667254
162543,9310439493,2023-12-28,1000059704,4566,6,553.8646,Familia C1,158.404580,157.536304,51,87,1,28.0,0.462051,-0.827775,6991.825440,0.846687,3849.646882
162544,9310439494,2023-12-28,1000059704,4566,3,276.9166,Familia C1,158.404580,157.536304,52,87,1,0.0,-1.187257,-1.005512,6991.825440,0.846687,3849.646882


In [6]:
# Dataset de entrenamiento con targets futuros y features temporales sin fuga
import math
import numpy as np

CONFIG_DATASET_MODELO = {
    'horizonte_fidelizacion_dias': 365,
    'peso_crecimiento_gasto': 0.6,
    'peso_crecimiento_frecuencia': 0.4,
    'escala_potencial': 1.0,
    'ventanas_historicas_dias': [30, 90, 180, 365],
}

COLUMNAS_ID_MODELO = [
    'Num.Fact',
    'Fecha',
    'Id. Cliente',
    'Id. Producto',
]

COLUMNAS_FEATURES_CANDIDATAS = [
    'Unidades',
    'Valores_H',
    'Bloque analítico',
    'Categoria_H',
    'Familia_H',
    'tipo_producto_binario',
    'mes_compra',
    'trimestre_compra',
    'anio_compra',
    'dias_desde_primera_compra_cliente',
    'dias_desde_compra_anterior_producto',
    'numero_compras_anteriores_producto',
    'numero_compras_anteriores_cliente',
    'numero_compras_anteriores_cliente_otros_productos',
    'numero_devoluciones_anteriores_producto',
    'gasto_anual_real_cliente_producto_hasta_fecha',
    'gasto_anual_real_cliente_categoria_hasta_fecha',
]
for ventana in CONFIG_DATASET_MODELO['ventanas_historicas_dias']:
    COLUMNAS_FEATURES_CANDIDATAS.extend([
        f'compras_cliente_{ventana}d_previas',
        f'gasto_cliente_{ventana}d_previo',
        f'compras_cliente_producto_{ventana}d_previas',
        f'gasto_cliente_producto_{ventana}d_previo',
        f'compras_cliente_categoria_{ventana}d_previas',
        f'gasto_cliente_categoria_{ventana}d_previo',
    ])

ventas_modelo_base = ventas_enriquecidas.copy()
ventas_modelo_base['_orden_modelo'] = range(len(ventas_modelo_base))
ventas_modelo_base['Fecha'] = pd.to_datetime(ventas_modelo_base['Fecha'])
ventas_modelo_base['_es_compra_valida'] = (
    (ventas_modelo_base['Unidades'] > 0)
    & (ventas_modelo_base['Valores_H'] > 0)
)
ventas_modelo_base['_es_devolucion'] = ventas_modelo_base['Valores_H'] <= 0
ventas_modelo_base['tipo_producto_binario'] = (
    ventas_modelo_base['Bloque analítico']
    .map({'Commodities': 0, 'Productos Técnicos': 1})
    .fillna(-1)
    .astype(int)
)

ventas_modelo_base = ventas_modelo_base.sort_values(
    ['Id. Cliente', 'Id. Producto', 'Fecha', 'Num.Fact', '_orden_modelo']
).copy()
ventas_modelo_base['fecha_compra_anterior_producto'] = (
    ventas_modelo_base['Fecha']
    .where(ventas_modelo_base['_es_compra_valida'])
    .groupby([ventas_modelo_base['Id. Cliente'], ventas_modelo_base['Id. Producto']])
    .transform(lambda s: s.shift().ffill())
)
ventas_modelo_base['dias_desde_compra_anterior_producto'] = (
    ventas_modelo_base['Fecha'] - ventas_modelo_base['fecha_compra_anterior_producto']
).dt.days.fillna(-1)
ventas_modelo_base['_compra_valida_int'] = ventas_modelo_base['_es_compra_valida'].astype(int)
ventas_modelo_base['numero_compras_anteriores_producto'] = (
    ventas_modelo_base
    .groupby(['Id. Cliente', 'Id. Producto'])['_compra_valida_int']
    .cumsum()
    - ventas_modelo_base['_compra_valida_int']
)
ventas_modelo_base['numero_compras_anteriores_cliente'] = (
    ventas_modelo_base
    .groupby('Id. Cliente')['_compra_valida_int']
    .cumsum()
    - ventas_modelo_base['_compra_valida_int']
)
ventas_modelo_base['numero_compras_anteriores_cliente_otros_productos'] = (
    ventas_modelo_base['numero_compras_anteriores_cliente']
    - ventas_modelo_base['numero_compras_anteriores_producto']
)
ventas_modelo_base['_devolucion_int'] = ventas_modelo_base['_es_devolucion'].astype(int)
ventas_modelo_base['numero_devoluciones_anteriores_producto'] = (
    ventas_modelo_base
    .groupby(['Id. Cliente', 'Id. Producto'])['_devolucion_int']
    .cumsum()
    - ventas_modelo_base['_devolucion_int']
)

ventas_modelo_base['mes_compra'] = ventas_modelo_base['Fecha'].dt.month
ventas_modelo_base['trimestre_compra'] = ventas_modelo_base['Fecha'].dt.quarter
ventas_modelo_base['anio_compra'] = ventas_modelo_base['Fecha'].dt.year
primera_compra_cliente = (
    ventas_modelo_base[ventas_modelo_base['_es_compra_valida']]
    .groupby('Id. Cliente')['Fecha']
    .transform('min')
)
ventas_modelo_base.loc[ventas_modelo_base['_es_compra_valida'], '_primera_compra_cliente'] = primera_compra_cliente
ventas_modelo_base['_primera_compra_cliente'] = (
    ventas_modelo_base
    .groupby('Id. Cliente')['_primera_compra_cliente']
    .ffill()
)
ventas_modelo_base['dias_desde_primera_compra_cliente'] = (
    ventas_modelo_base['Fecha'] - ventas_modelo_base['_primera_compra_cliente']
).dt.days.fillna(-1)

ventas_modelo_base['_valor_compra_valida'] = ventas_modelo_base['Valores_H'].where(
    ventas_modelo_base['_es_compra_valida'],
    0
)
ventas_modelo_base['gasto_anual_real_cliente_producto_hasta_fecha'] = (
    ventas_modelo_base
    .groupby(['Id. Cliente', 'Id. Producto', 'anio_compra'])['_valor_compra_valida']
    .cumsum()
)
ventas_modelo_base['gasto_anual_real_cliente_categoria_hasta_fecha'] = (
    ventas_modelo_base
    .groupby(['Id. Cliente', 'Categoria_H', 'anio_compra'])['_valor_compra_valida']
    .cumsum()
)

compras_validas_features = ventas_modelo_base[ventas_modelo_base['_es_compra_valida']].copy()
compras_validas_features = compras_validas_features.sort_values(
    ['Id. Cliente', 'Fecha', 'Num.Fact', '_orden_modelo']
).copy()

def add_rolling_history(df, group_cols, prefix, ventanas):
    salida = []
    for _, grupo in df.groupby(group_cols, sort=False, dropna=False):
        grupo = grupo.sort_values(['Fecha', 'Num.Fact', '_orden_modelo']).copy()
        fechas = grupo['Fecha'].astype('int64').to_numpy() // 86_400_000_000_000
        valores = grupo['Valores_H'].to_numpy(dtype=float)
        cumsum = np.concatenate([[0.0], np.cumsum(valores)])
        posiciones_fin = np.arange(len(grupo))
        for ventana in ventanas:
            posiciones_inicio = np.searchsorted(fechas, fechas - ventana, side='left')
            grupo[f'compras_{prefix}_{ventana}d_previas'] = posiciones_fin - posiciones_inicio
            grupo[f'gasto_{prefix}_{ventana}d_previo'] = cumsum[posiciones_fin] - cumsum[posiciones_inicio]
        salida.append(grupo)
    return pd.concat(salida).sort_index()

ventanas = CONFIG_DATASET_MODELO['ventanas_historicas_dias']
rolling_cliente = add_rolling_history(compras_validas_features, ['Id. Cliente'], 'cliente', ventanas)
rolling_cliente_producto = add_rolling_history(compras_validas_features, ['Id. Cliente', 'Id. Producto'], 'cliente_producto', ventanas)
rolling_cliente_categoria = add_rolling_history(compras_validas_features, ['Id. Cliente', 'Categoria_H'], 'cliente_categoria', ventanas)

cols_rolling = []
for ventana in ventanas:
    cols_rolling.extend([
        f'compras_cliente_{ventana}d_previas',
        f'gasto_cliente_{ventana}d_previo',
        f'compras_cliente_producto_{ventana}d_previas',
        f'gasto_cliente_producto_{ventana}d_previo',
        f'compras_cliente_categoria_{ventana}d_previas',
        f'gasto_cliente_categoria_{ventana}d_previo',
    ])

compras_validas_features = compras_validas_features.join(
    rolling_cliente[[c for c in rolling_cliente.columns if c.startswith(('compras_cliente_', 'gasto_cliente_'))]]
)
compras_validas_features = compras_validas_features.join(
    rolling_cliente_producto[[c for c in rolling_cliente_producto.columns if c.startswith(('compras_cliente_producto_', 'gasto_cliente_producto_'))]]
)
compras_validas_features = compras_validas_features.join(
    rolling_cliente_categoria[[c for c in rolling_cliente_categoria.columns if c.startswith(('compras_cliente_categoria_', 'gasto_cliente_categoria_'))]]
)

compras_validas_features = compras_validas_features.sort_values(
    ['Id. Cliente', 'Id. Producto', 'Fecha', 'Num.Fact', '_orden_modelo']
).copy()
compras_validas_features['fecha_proxima_compra_producto'] = (
    compras_validas_features
    .groupby(['Id. Cliente', 'Id. Producto'])['Fecha']
    .shift(-1)
)
compras_validas_features['vuelve_a_comprar'] = (
    compras_validas_features['fecha_proxima_compra_producto'].notna().astype(int)
)
compras_validas_features['dias_hasta_proxima_compra'] = (
    compras_validas_features['fecha_proxima_compra_producto']
    - compras_validas_features['Fecha']
).dt.days
compras_validas_features['dias_hasta_proxima_compra'] = (
    compras_validas_features['dias_hasta_proxima_compra']
    .fillna(1500)
    .clip(lower=0)
    .astype(int)
)

def calcular_target_potencial_cliente(grupo):
    grupo = grupo.sort_values(['Fecha', 'Num.Fact', '_orden_modelo']).copy()
    fechas = grupo['Fecha']
    valores = grupo['Valores_H']
    horizonte = pd.Timedelta(days=CONFIG_DATASET_MODELO['horizonte_fidelizacion_dias'])
    dias_horizonte = CONFIG_DATASET_MODELO['horizonte_fidelizacion_dias']
    peso_gasto = CONFIG_DATASET_MODELO['peso_crecimiento_gasto']
    peso_frecuencia = CONFIG_DATASET_MODELO['peso_crecimiento_frecuencia']
    escala = CONFIG_DATASET_MODELO['escala_potencial']

    targets = []
    gasto_base_anual = []
    gasto_futuro_anual = []
    frecuencia_base_anual = []
    frecuencia_futura_anual = []

    for _, fila in grupo.iterrows():
        fecha = fila['Fecha']
        mascara_base = (fechas >= fecha - horizonte) & (fechas <= fecha)
        mascara_futuro = (fechas > fecha) & (fechas <= fecha + horizonte)

        gasto_base = valores[mascara_base].sum() * 365 / dias_horizonte
        gasto_futuro = valores[mascara_futuro].sum() * 365 / dias_horizonte
        freq_base = mascara_base.sum() * 365 / dias_horizonte
        freq_futura = mascara_futuro.sum() * 365 / dias_horizonte

        gasto_base_anual.append(gasto_base)
        gasto_futuro_anual.append(gasto_futuro)
        frecuencia_base_anual.append(freq_base)
        frecuencia_futura_anual.append(freq_futura)

        if freq_futura == 0:
            targets.append(-1.0)
            continue

        crecimiento_gasto = (gasto_futuro - gasto_base) / max(abs(gasto_base), 1e-9)
        crecimiento_frecuencia = (freq_futura - freq_base) / max(abs(freq_base), 1e-9)
        score = (peso_gasto * crecimiento_gasto) + (peso_frecuencia * crecimiento_frecuencia)
        targets.append(float(math.tanh(score / escala)))

    grupo['gasto_base_anual_fidelizacion'] = gasto_base_anual
    grupo['gasto_futuro_anual_fidelizacion'] = gasto_futuro_anual
    grupo['frecuencia_base_anual_fidelizacion'] = frecuencia_base_anual
    grupo['frecuencia_futura_anual_fidelizacion'] = frecuencia_futura_anual
    grupo['target_potencial_cliente'] = targets
    return grupo

compras_validas_features = (
    compras_validas_features
    .groupby(['Id. Cliente', 'Id. Producto'], group_keys=False)
    .apply(calcular_target_potencial_cliente)
)

COLUMNAS_TARGET = [
    'vuelve_a_comprar',
    'dias_hasta_proxima_compra',
    'target_potencial_cliente',
]
COLUMNAS_AUDITORIA_TARGET = [
    'gasto_base_anual_fidelizacion',
    'gasto_futuro_anual_fidelizacion',
    'frecuencia_base_anual_fidelizacion',
    'frecuencia_futura_anual_fidelizacion',
]
COLUMNAS_FEATURES_MODELO = [
    columna for columna in COLUMNAS_FEATURES_CANDIDATAS
    if columna in compras_validas_features.columns
]
COLUMNAS_ID_MODELO = [
    columna for columna in COLUMNAS_ID_MODELO
    if columna in compras_validas_features.columns
]

dataset_modelo = (
    compras_validas_features
    .sort_values('_orden_modelo')
    [COLUMNAS_ID_MODELO + COLUMNAS_FEATURES_MODELO + COLUMNAS_TARGET + COLUMNAS_AUDITORIA_TARGET]
    .replace([np.inf, -np.inf], np.nan)
    .fillna({col: 0 for col in COLUMNAS_FEATURES_MODELO if col not in ['Bloque analítico', 'Categoria_H', 'Familia_H']})
    .reset_index(drop=True)
)

ruta_dataset_modelo_csv = 'dataset_modelo.csv'
dataset_modelo.to_csv(ruta_dataset_modelo_csv, index=False, encoding='utf-8-sig')

print(f'Dataset exportado a: {ruta_dataset_modelo_csv}')
dataset_modelo




C:\Users\PE2554\AppData\Local\Temp\ipykernel_5072\1543460801.py:252: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calcular_target_potencial_cliente)
C:\Users\PE2554\AppData\Local\Temp\ipykernel_5072\1543460801.py:279: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace([np.inf, -np.inf], np.nan)


Dataset exportado a: dataset_modelo.csv


,Num.Fact,Fecha,Id. Cliente,Id. Producto,Unidades,Valores_H,Bloque analítico,Categoria_H,Familia_H,tipo_producto_binario,...,gasto_cliente_producto_365d_previo,compras_cliente_categoria_365d_previas,gasto_cliente_categoria_365d_previo,vuelve_a_comprar,dias_hasta_proxima_compra,target_potencial_cliente,gasto_base_anual_fidelizacion,gasto_futuro_anual_fidelizacion,frecuencia_base_anual_fidelizacion,frecuencia_futura_anual_fidelizacion
0,3910002518,2024-01-04,1818,3408,4,284.6724,Commodities,Categoria C1,Familia C1,0,...,211.3848,5,786.9782,1,98,0.240755,496.0572,533.7372,2.0,3.0
1,3910002518,2024-01-04,1818,4565,1,114.6728,Commodities,Categoria C1,Familia C1,0,...,255.4704,6,1071.6506,0,1500,-1.000000,370.1432,0.0000,3.0,0.0
2,3910002518,2024-01-04,1818,4567,1,146.5752,Commodities,Categoria C1,Familia C1,0,...,320.1230,7,1186.3234,1,98,0.106252,466.6982,549.6570,3.0,3.0
3,3910002519,2024-01-04,5787,5099,1,335.9172,Productos Técnicos,Categoria T1,Familia T1,1,...,1251.4156,3,1728.2874,1,83,0.567870,1587.3328,2586.5750,3.0,5.0
4,3910002520,2024-01-04,1000080471,3408,6,384.3046,Commodities,Categoria C1,Familia C1,0,...,880.7700,7,1451.6220,1,76,-0.285553,1265.0746,1067.4744,4.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156916,9310439486,2023-12-28,33953,4567,5,533.5488,Commodities,Categoria C1,Familia C1,0,...,5015.3650,18,8268.5620,1,63,-0.010565,5548.9138,6376.0212,8.0,6.0
156917,9310439486,2023-12-28,33953,4912,24,756.6144,Commodities,Categoria C2,Familia C2,0,...,5994.8880,10,9716.1648,1,63,-0.216467,6751.5024,5401.8048,8.0,6.0
156918,9310439492,2023-12-28,41637,5112,2,688.4450,Productos Técnicos,Categoria T1,Familia T2,1,...,0.0000,0,0.0000,0,1500,-1.000000,688.4450,0.0000,1.0,0.0
156919,9310439493,2023-12-28,1000059704,4566,6,553.8646,Commodities,Categoria C1,Familia C1,0,...,8019.3402,26,9210.1852,1,0,-0.158310,8850.1214,7085.2216,20.0,18.0


In [7]:
# Columnas de dataset_modelo con valores NaN
columnas_con_nan = (
    dataset_modelo
    .isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .rename('n_nan')
    .reset_index()
    .rename(columns={'index': 'columna'})
)
columnas_con_nan['pct_nan'] = (
    columnas_con_nan['n_nan'] / len(dataset_modelo) * 100
).round(2)

columnas_con_nan


,columna,n_nan,pct_nan
